<a href="https://colab.research.google.com/github/weirdrazak/climate-financing-model-analysis/blob/main/Political_Economy_of_Climate_Financing_in_Africa_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
"""
WACC and LCOE Analysis: West African Solar PV Cluster
=======================================================

Reproduces every calculation and chart used in the thesis:
  1. Country-specific WACC (CAPM-based, following Damodaran / Egli et al. 2019)
  2. Baseline LCOE under country-specific vs. uniform WACC
  3. Scenario analysis: LCOE after a 3-percentage-point policy-driven WACC reduction
  4. All associated charts (Figure 4.1 style combined chart, and standalone
     WACC line / LCOE bar charts, for both the baseline and the 3pp scenario)

Technical/financial parameters (CAPEX, O&M, lifetime, capacity factor) are
sourced directly from Bogdanov et al. (2019), Supplementary Tables 2 and 5,
for PV single-axis tracking in the "WS" subregion (Ghana, Cote d'Ivoire,
Benin, Burkina Faso, Togo). The default year is 2030 (see BOGDANOV_PARAMS
below); switch YEAR to 2050 to reproduce the robustness-check scenario.

Requirements: matplotlib, numpy
    pip install matplotlib numpy --break-system-packages
"""

import numpy as np
import matplotlib.pyplot as plt

 =====================================================================
# 1. INPUT PARAMETERS
# =====================================================================

In [14]:
# ---- CAPM inputs (shared across all countries) ----
RF = 2.48          # Risk-free rate (%), FRED DGS10, annual average 2015-2024
ERP = 4.23         # Mature market equity risk premium (%), Damodaran, Jan 2026
BETA_U = 0.54      # Unlevered beta for renewable energy sector, Damodaran Global Betas
D_V = 0.60         # Debt share of capital structure (D/V)
E_V = 1 - D_V      # Equity share of capital structure (E/V)
D_E = D_V / E_V    # Debt-to-equity ratio
LENDER_MARKUP = 1.00  # Project lender markup over sovereign spread (%)

# ---- Country-specific inputs (Moody's rating, CRP, default spread, tax rate) ----
# CRP = Country Risk Premium (feeds into Ke); Default Spread feeds into Kd.
# Source: Damodaran country risk premium tables, Moody's sovereign ratings.
COUNTRIES = {
    "Ghana":          {"rating": "Caa1", "crp": 9.71, "spread": 6.37, "tax": 25.0},
    "Burkina Faso":   {"rating": "Caa1", "crp": 9.71, "spread": 6.37, "tax": 27.5},
    "Togo":           {"rating": "B3",   "crp": 8.41, "spread": 5.52, "tax": 27.0},
    "Benin":          {"rating": "B1",   "crp": 5.83, "spread": 3.83, "tax": 30.0},
    "Cote d'Ivoire":  {"rating": "Ba2",  "crp": 3.90, "spread": 2.56, "tax": 25.0},
}

# ---- Comparators ----
GERMANY = {"rating": "Aaa", "crp": 0.00, "spread": 0.00, "tax": 30.0}
UNIFORM_WACC = 7.00   # Uniform WACC assumed by Bogdanov et al. (2019), main text

# ---- Bogdanov et al. (2019) Supplementary Table 5: PV single-axis tracking ----
# Format: year -> (CAPEX [EUR/kW,el], O&M fixed [EUR/kW,el/yr], lifetime [years])
BOGDANOV_PARAMS = {
    2015: (1150, 17.3, 30),
    2020: (638,  15.0, 30),
    2025: (513,  13.0, 35),
    2030: (429,  12.0, 35),
    2035: (371,  11.0, 35),
    2040: (330,  10.0, 40),
    2045: (297,   9.0, 40),
    2050: (271,   8.0, 40),
}

# ---- Bogdanov et al. (2019) Supplementary Table 2: capacity factor ----
# Subregion "WS" = Ghana, Cote d'Ivoire, Benin, Burkina Faso, Togo
# PV single-axis: 2099 full-load-hours/year (not year-dependent)
FULL_LOAD_HOURS = 2099
CAPACITY_FACTOR = FULL_LOAD_HOURS / 8760  # = 0.2396 (23.96%)

# ---- Which year's parameter set to use for LCOE (primary scenario) ----
YEAR = 2030
CAPEX, OM, LIFETIME = BOGDANOV_PARAMS[YEAR]

# ---- Policy intervention scenario ----
WACC_REDUCTION_PP = 3.0   # 3-percentage-point combined intervention
                          # (2pp downside-risk mitigation + 1pp upside-risk enhancement)


# =====================================================================
# 2. WACC CALCULATION (CAPM-based)
# =====================================================================

In [15]:

def compute_wacc(crp, spread, tax_pct, rf=RF, erp=ERP, beta_u=BETA_U,
                  d_v=D_V, e_v=E_V, d_e=D_E, markup=LENDER_MARKUP, verbose=False):
    """
    Compute WACC using the CAPM framework:
        Beta_levered = Beta_u * [1 + (1 - t) * D/E]           (Hamada equation)
        Ke           = Rf + Beta_levered * ERP + CRP          (cost of equity)
        Kd_pretax    = Rf + default_spread + lender_markup    (cost of debt, pre-tax)
        Kd_aftertax  = Kd_pretax * (1 - t)                    (tax shield applied)
        WACC         = E/V * Ke + D/V * Kd_aftertax
    """
    t = tax_pct / 100
    beta_lev = beta_u * (1 + (1 - t) * d_e)
    ke = rf + beta_lev * erp + crp
    kd_pretax = rf + spread + markup
    kd_aftertax = kd_pretax * (1 - t)
    wacc = e_v * ke + d_v * kd_aftertax

    if verbose:
        print(f"  Beta_levered = {beta_lev:.4f}")
        print(f"  Ke           = {ke:.4f}%")
        print(f"  Kd (pretax)  = {kd_pretax:.4f}%")
        print(f"  Kd (aftertax)= {kd_aftertax:.4f}%")
        print(f"  WACC         = {wacc:.4f}%")

    return wacc, ke, kd_aftertax, beta_lev


def compute_all_wacc():
    """Return {country: wacc} for all countries plus Germany."""
    results = {}
    for name, d in COUNTRIES.items():
        wacc, *_ = compute_wacc(d["crp"], d["spread"], d["tax"])
        results[name] = wacc
    results["Germany"] = compute_wacc(GERMANY["crp"], GERMANY["spread"], GERMANY["tax"])[0]
    return results


# =====================================================================
# 3. LCOE CALCULATION
# =====================================================================

In [16]:

def capital_recovery_factor(rate_pct, years):
    """CRF = r(1+r)^T / [(1+r)^T - 1]"""
    r = rate_pct / 100
    return r * (1 + r) ** years / ((1 + r) ** years - 1)


def compute_lcoe(rate_pct, capex=CAPEX, om=OM, lifetime=LIFETIME, cf=CAPACITY_FACTOR):
    """
    LCOE = (CAPEX * CRF + O&M) / AEP
    where AEP (Annual Energy Production, MWh/kW/yr) = capacity_factor * 8760 / 1000
    """
    crf = capital_recovery_factor(rate_pct, lifetime)
    aep = cf * 8760 / 1000
    return (capex * crf + om) / aep


def compute_baseline_lcoe_table(wacc_dict):
    """Return {country: lcoe} for all countries, Germany, and the uniform benchmark."""
    lcoe_uniform = compute_lcoe(UNIFORM_WACC)
    table = {"Uniform (Bogdanov et al. 2019)": lcoe_uniform}
    for name, wacc in wacc_dict.items():
        table[name] = compute_lcoe(wacc)
    return table


# =====================================================================
# 4. SCENARIO ANALYSIS: 3-PERCENTAGE-POINT WACC REDUCTION
# =====================================================================


In [17]:

def compute_scenario_table(wacc_dict, reduction_pp=WACC_REDUCTION_PP):
    """
    For each country, compute baseline WACC/LCOE, post-intervention WACC/LCOE,
    and the resulting LCOE reduction (absolute and percentage).
    """
    scenario = {}
    for name, wacc_base in wacc_dict.items():
        if name == "Germany":
            continue
        wacc_after = wacc_base - reduction_pp
        lcoe_base = compute_lcoe(wacc_base)
        lcoe_after = compute_lcoe(wacc_after)
        reduction_abs = lcoe_after - lcoe_base
        reduction_pct = (lcoe_after / lcoe_base - 1) * 100
        scenario[name] = {
            "wacc_base": wacc_base,
            "wacc_after": wacc_after,
            "lcoe_base": lcoe_base,
            "lcoe_after": lcoe_after,
            "reduction_abs": reduction_abs,
            "reduction_pct": reduction_pct,
        }
    return scenario



# =====================================================================
# 5. PRINT RESULTS (Tables 4.1, 4.2, 4.3 equivalents)
# =====================================================================


In [18]:

def print_all_tables():
    wacc = compute_all_wacc()
    lcoe_table = compute_baseline_lcoe_table(wacc)
    scenario = compute_scenario_table(wacc)

    print("=" * 70)
    print(f"TABLE 4.1: Country-specific WACC decomposition ({YEAR} parameters)")
    print("=" * 70)
    for name, d in COUNTRIES.items():
        w, ke, kd, beta = compute_wacc(d["crp"], d["spread"], d["tax"])
        print(f"{name:15} Rating={d['rating']:5} CRP={d['crp']:5.2f}%  "
              f"Spread={d['spread']:5.2f}%  Tax={d['tax']:5.1f}%  WACC={w:.2f}%")
    g_w = wacc["Germany"]
    print(f"{'Germany':15} Rating={GERMANY['rating']:5} CRP={GERMANY['crp']:5.2f}%  "
          f"Spread={GERMANY['spread']:5.2f}%  Tax={GERMANY['tax']:5.1f}%  WACC={g_w:.2f}%")
    print(f"{'Uniform benchmark':15} WACC={UNIFORM_WACC:.2f}%")

    print()
    print("=" * 70)
    print(f"TABLE 4.2: Baseline LCOE (EUR/MWh) — CAPEX={CAPEX}, O&M={OM}, "
          f"Lifetime={LIFETIME}yr, CF={CAPACITY_FACTOR:.4f}")
    print("=" * 70)
    lcoe_uniform = lcoe_table["Uniform (Bogdanov et al. 2019)"]
    for name in list(COUNTRIES.keys()) + ["Germany"]:
        l = lcoe_table[name]
        pct = (l / lcoe_uniform - 1) * 100
        print(f"{name:15} LCOE={l:6.2f}  ({pct:+.1f}% vs uniform)")
    print(f"{'Uniform':15} LCOE={lcoe_uniform:6.2f}")

    print()
    print("=" * 70)
    print(f"TABLE 4.3: Effect of a {WACC_REDUCTION_PP:.0f}pp WACC reduction")
    print("=" * 70)
    for name, s in scenario.items():
        print(f"{name:15} WACC {s['wacc_base']:.2f}%->{s['wacc_after']:.2f}%  "
              f"LCOE {s['lcoe_base']:.2f}->{s['lcoe_after']:.2f}  "
              f"({s['reduction_abs']:+.2f}, {s['reduction_pct']:+.1f}%)")

    return wacc, lcoe_table, scenario



# =====================================================================
# 6. CHARTS
# =====================================================================

In [22]:

RATINGS = {name: d["rating"] for name, d in COUNTRIES.items()}
COUNTRY_ORDER = list(COUNTRIES.keys())  # Ghana, Burkina Faso, Togo, Benin, Cote d'Ivoire


def chart_figure_4_1(wacc, lcoe_table, save_path="Figure_4_1.png"):
    """Combined two-panel chart: (a) WACC ranking, (b) LCOE comparison."""
    wacc_vals = [wacc[c] for c in COUNTRY_ORDER]
    lcoe_vals = [lcoe_table[c] for c in COUNTRY_ORDER]
    lcoe_uniform = lcoe_table["Uniform (Bogdanov et al. 2019)"]
    pct_change = [(l / lcoe_uniform - 1) * 100 for l in lcoe_vals]

    fig, axes = plt.subplots(2, 1, figsize=(8.5, 10), gridspec_kw={"height_ratios": [1, 1.3]})
    fig.subplots_adjust(hspace=0.5)

    # Panel a: WACC line chart
    ax = axes[0]
    x = np.arange(len(COUNTRY_ORDER))
    ax.plot(x, wacc_vals, color="#E07B39", marker="o", linewidth=2, zorder=3,
            label="Country-specific WACC, reflecting sovereign risk")
    ax.axhline(UNIFORM_WACC, color="gray", linewidth=1.5, linestyle="--",
               label=f"Uniform WACC assumed by Bogdanov et al. (2019), {UNIFORM_WACC:.1f}%")
    ax.axhline(wacc["Germany"], color="#2E7D32", linewidth=1.8,
               label=f"Best-in-class CoC (Germany, {wacc['Germany']:.2f}%)")
    offsets = [12, 12, -30, 12, 12]
    va_list = ["bottom", "bottom", "top", "bottom", "bottom"]
    for xi, yi, c, off, va in zip(x, wacc_vals, COUNTRY_ORDER, offsets, va_list):
        ax.annotate(f"{c}: {yi:.2f}%\n({RATINGS[c]})", (xi, yi), textcoords="offset points",
                    xytext=(0, off), ha="center", va=va, fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(COUNTRY_ORDER, fontsize=8)
    ax.set_xlim(-0.5, len(COUNTRY_ORDER) - 0.2)
    ax.set_ylabel("Estimated WACC of\nsolar PV projects")
    ax.set_ylim(0, 14)
    ax.yaxis.set_major_formatter(lambda v, pos: f"{int(v)}%")
    ax.set_title("a", loc="left", fontweight="bold", fontsize=13)
    ax.legend(fontsize=7, loc="upper right", frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

    # Panel b: LCOE bar chart
    ax2 = axes[1]
    width = 0.35
    bars1 = ax2.bar(x - width/2, [lcoe_uniform]*len(COUNTRY_ORDER), width,
                     color="none", edgecolor="gray", hatch="////",
                     label=f"Uniform WACC ({UNIFORM_WACC:.1f}%, Bogdanov et al. 2019)")
    bars2 = ax2.bar(x + width/2, lcoe_vals, width, color="#C0392B",
                     label="Country-specific WACC")
    for b in bars1:
        ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.4, f"{b.get_height():.1f}",
                  ha="center", fontsize=8.5, color="dimgray")
    for b, pc in zip(bars2, pct_change):
        ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.4, f"{b.get_height():.1f}",
                  ha="center", fontsize=8.5)
        ax2.annotate(f"+{pc:.0f}%", (b.get_x()+b.get_width()/2, b.get_height()+2.0),
                     ha="center", fontsize=9, fontweight="bold",
                     bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", lw=0.8))
    ax2.set_xticks(x); ax2.set_xticklabels(COUNTRY_ORDER, fontsize=8)
    ax2.set_ylabel("LCOE of solar PV\n(EUR/MWh)")
    ax2.set_ylim(0, max(lcoe_vals) * 1.35)
    ax2.set_title("b", loc="left", fontweight="bold", fontsize=13)
    ax2.legend(fontsize=8, loc="upper left", frameon=False)
    ax2.spines[["top", "right"]].set_visible(False)

    fig.suptitle(f"Country-specific WACC and its effect on {YEAR} LCOE of solar PV\n"
                 f"West African cluster (Bogdanov et al. 2019 assumptions)",
                 fontsize=9.5, y=0.998)
    fig.text(0.5, 0.005,
              f"LCOE assumptions (Bogdanov et al. 2019, {YEAR}): CAPEX EUR{CAPEX}/kW, "
              f"O&M EUR{OM}/kW/yr, lifetime {LIFETIME} yrs; capacity factor "
              f"{CAPACITY_FACTOR*100:.2f}% / {FULL_LOAD_HOURS} FLh/yr (subregion WS).",
              ha="center", fontsize=6.3, style="italic", color="dimgray")

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


def chart_wacc_line(wacc, save_path="Chart_WACC_baseline.png", scenario=None):
    """
    Standalone WACC line chart. If `scenario` is provided (from compute_scenario_table),
    also plots the post-intervention WACC line.
    """
    wacc_vals = [wacc[c] for c in COUNTRY_ORDER]
    x = np.arange(len(COUNTRY_ORDER))

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.plot(x, wacc_vals, color="#E07B39",
            marker="o", markersize=10, linewidth=3, zorder=3,
            label="Baseline WACC (country-specific)" if scenario else
                  "Country-specific WACC, reflecting sovereign risk")

    if not scenario:
        ax.axhline(UNIFORM_WACC, color="gray", linewidth=2.2, linestyle="--",
                   label=f"Uniform WACC assumed by Bogdanov et al. (2019), {UNIFORM_WACC:.1f}%")

    ax.axhline(wacc["Germany"], color="#2E7D32", linewidth=2.5,
               label=f"Best-in-class CoC (Germany, {wacc['Germany']:.2f}%)")

    offsets = [22, 22, -38, 22, 22]
    va_list = ["bottom", "bottom", "top", "bottom", "bottom"]
    for xi, yi, c, off, va in zip(x, wacc_vals, COUNTRY_ORDER, offsets, va_list):
        ax.annotate(f"{c}: {yi:.2f}%\n({RATINGS[c]})", (xi, yi), textcoords="offset points",
                    xytext=(0, off), ha="center", va=va, fontsize=13.5, fontweight="bold")

    if scenario:
        wacc_after = [scenario[c]["wacc_after"] for c in COUNTRY_ORDER]
        ax.plot(x, wacc_after, color="#2980B9", marker="o", markersize=10, linewidth=3,
                zorder=3, label="WACC after 3pp policy intervention")
        for xi, yi in zip(x, wacc_after):
            ax.annotate(f"{yi:.2f}%", (xi, yi), textcoords="offset points",
                        xytext=(0, -22), ha="center", fontsize=12,
                        color="#2980B9", fontweight="bold")

    ax.set_xticks(x); ax.set_xticklabels(COUNTRY_ORDER, fontsize=15, fontweight="bold")
    ax.set_ylabel("Estimated WACC of solar PV projects", fontsize=15, fontweight="bold")
    ax.set_ylim(0, 14)
    ax.set_xlim(-0.4, len(COUNTRY_ORDER) - 0.2)
    ax.yaxis.set_major_formatter(lambda v, pos: f"{int(v)}%")
    ax.tick_params(axis="y", labelsize=13)
    title = ("Effect of a 3-Percentage-Point Policy Intervention on WACC"
              if scenario else "Country-Specific WACC Decomposition")
    ax.set_title(f"{title}\nWest African Cluster and Comparators", fontsize=17,
                 fontweight="bold", pad=22)
    ax.legend(fontsize=12.5, loc="upper right", frameon=True, framealpha=0.95)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle=":", alpha=0.4)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


def chart_lcoe_bar(lcoe_table, save_path="Chart_LCOE_baseline.png", scenario=None):
    """
    Standalone LCOE bar chart. If `scenario` is provided, plots baseline vs.
    post-intervention LCOE instead of uniform vs. country-specific.
    """
    x = np.arange(len(COUNTRY_ORDER))
    width = 0.35
    fig, ax = plt.subplots(figsize=(12, 8))

    if scenario:
        base_vals = [scenario[c]["lcoe_base"] for c in COUNTRY_ORDER]
        after_vals = [scenario[c]["lcoe_after"] for c in COUNTRY_ORDER]
        pct_vals = [scenario[c]["reduction_pct"] for c in COUNTRY_ORDER]
        bars1 = ax.bar(x - width/2, base_vals, width, color="#C0392B", label="Baseline LCOE")
        bars2 = ax.bar(x + width/2, after_vals, width, color="#2980B9",
                        label="LCOE after 3pp intervention")
        for b in bars1:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.6, f"{b.get_height():.2f}",
                    ha="center", fontsize=13.5, fontweight="bold")
        for b, pc in zip(bars2, pct_vals):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.6, f"{b.get_height():.2f}",
                    ha="center", fontsize=13.5, fontweight="bold", color="#2980B9")
            ax.annotate(f"{pc:.1f}%", (b.get_x()+b.get_width()/2, b.get_height()+3.2),
                        ha="center", fontsize=13, fontweight="bold", color="#2980B9",
                        bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="#2980B9", lw=1.4))
        title = "Effect of a 3-Percentage-Point WACC Reduction on LCOE"
        max_val = max(base_vals)
    else:
        lcoe_uniform = lcoe_table["Uniform (Bogdanov et al. 2019)"]
        country_vals = [lcoe_table[c] for c in COUNTRY_ORDER]
        pct_vals = [(l/lcoe_uniform - 1)*100 for l in country_vals]
        bars1 = ax.bar(x - width/2, [lcoe_uniform]*len(COUNTRY_ORDER), width,
                        color="none", edgecolor="gray", hatch="////",
                        label=f"Uniform WACC ({UNIFORM_WACC:.1f}%, Bogdanov et al. 2019)")
        bars2 = ax.bar(x + width/2, country_vals, width, color="#C0392B",
                        label="Country-specific WACC")
        for b in bars1:
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.6, f"{b.get_height():.2f}",
                    ha="center", fontsize=13.5, fontweight="bold", color="dimgray")
        for b, pc in zip(bars2, pct_vals):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.6, f"{b.get_height():.2f}",
                    ha="center", fontsize=13.5, fontweight="bold")
            ax.annotate(f"+{pc:.1f}%", (b.get_x()+b.get_width()/2, b.get_height()+3.2),
                        ha="center", fontsize=13, fontweight="bold", color="#C0392B",
                        bbox=dict(boxstyle="round,pad=0.32", fc="white", ec="#C0392B", lw=1.4))
        title = "Effect of Country-Specific WACC on LCOE"
        max_val = max(country_vals)

    ax.axhline(lcoe_table["Germany"], color="#2E7D32", linewidth=2.2, linestyle="--",
               label=f"Germany best-in-class (EUR {lcoe_table['Germany']:.2f}/MWh)")
    ax.set_xticks(x); ax.set_xticklabels(COUNTRY_ORDER, fontsize=15, fontweight="bold")
    ax.set_ylabel("LCOE of solar PV (EUR/MWh)", fontsize=15, fontweight="bold")
    ax.set_ylim(0, max_val * 1.35)
    ax.tick_params(axis="y", labelsize=13)
    ax.set_title(f"{title}\nWest African Cluster ({YEAR}, Bogdanov et al. 2019 assumptions)",
                 fontsize=16.5, fontweight="bold", pad=22)
    ax.legend(fontsize=12.5, loc="upper right", frameon=True, framealpha=0.95)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", linestyle=":", alpha=0.35)

    # fig.text(0.5, 0.01,
    #           f"LCOE assumptions (Bogdanov et al. 2019, {YEAR}): CAPEX EUR{CAPEX}/kW, "
    #           f"O&M EUR{OM}/kW/yr, lifetime {LIFETIME} yrs, capacity factor "
    #           f"{CAPACITY_FACTOR*100:.2f}%.",
    #           ha="center", fontsize=9.5, style="italic", color="dimgray")

    plt.tight_layout(rect=[0, 0.035, 1, 1])
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.savefig(save_path.replace(".png", ".pdf"), bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")


# =====================================================================
# 7. MAIN
# =====================================================================


In [23]:

if __name__ == "__main__":
    wacc, lcoe_table, scenario = print_all_tables()

    # Combined Figure 4.1 (panel a: WACC, panel b: LCOE)
    chart_figure_4_1(wacc, lcoe_table, save_path="Figure_4_1.png")

    # Standalone baseline charts (before intervention)
    chart_wacc_line(wacc, save_path="Chart_WACC_baseline.png")
    chart_lcoe_bar(lcoe_table, save_path="Chart_LCOE_baseline.png")

    # Standalone 3pp-intervention charts
    chart_wacc_line(wacc, save_path="Chart_WACC_3pp.png", scenario=scenario)
    chart_lcoe_bar(lcoe_table, save_path="Chart_LCOE_3pp.png", scenario=scenario)

    print("\nAll charts saved to the current directory.")
    print(f"To reproduce the 2050 robustness check, set YEAR = 2050 at the top of this "
          f"script and re-run.")


TABLE 4.1: Country-specific WACC decomposition (2030 parameters)
Ghana           Rating=Caa1  CRP= 9.71%  Spread= 6.37%  Tax= 25.0%  WACC=11.25%
Burkina Faso    Rating=Caa1  CRP= 9.71%  Spread= 6.37%  Tax= 27.5%  WACC=11.07%
Togo            Rating=B3    CRP= 8.41%  Spread= 5.52%  Tax= 27.0%  WACC=10.21%
Benin           Rating=B1    CRP= 5.83%  Spread= 3.83%  Tax= 30.0%  WACC=8.27%
Cote d'Ivoire   Rating=Ba2   CRP= 3.90%  Spread= 2.56%  Tax= 25.0%  WACC=7.21%
Germany         Rating=Aaa   CRP= 0.00%  Spread= 0.00%  Tax= 30.0%  WACC=4.33%
Uniform benchmark WACC=7.00%

TABLE 4.2: Baseline LCOE (EUR/MWh) — CAPEX=429, O&M=12.0, Lifetime=35yr, CF=0.2396
Ghana           LCOE= 29.27  (+36.1% vs uniform)
Burkina Faso    LCOE= 28.93  (+34.5% vs uniform)
Togo            LCOE= 27.31  (+27.0% vs uniform)
Benin           LCOE= 23.73  (+10.4% vs uniform)
Cote d'Ivoire   LCOE= 21.87  (+1.7% vs uniform)
Germany         LCOE= 17.16  (-20.2% vs uniform)
Uniform         LCOE= 21.50

TABLE 4.3: Effect of a 